## Import necessary packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import hdf5plugin
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
from CellPLM.utils import set_seed
from CellPLM.pipeline.cell_embedding import CellEmbeddingPipeline
import scanpy as sc
import matplotlib.pyplot as plt
# import rapids_singlecell as rsc  # For faster evaluation, we recommend the installation of rapids_singlecell.

## Specify important parameters before getting started

In [ ]:
PRETRAIN_VERSION = '20231027_85M'
DEVICE = 'cuda:0'

## Load Downstream Dataset

In [ ]:
set_seed(42)

In [ ]:
import anndata

# Load the h5ad file
data = ad.read_h5ad('/media/rokny/DATA2/Sally/data/scRNA-seq/BMMC/GSE194122_cite_BMMC_processed.h5ad')

In [ ]:
# Move processed data to a new layer
data.layers['processed'] = data.X.copy()

# Replace .X with raw counts
data.X = data.layers['counts'].copy()

In [ ]:
# normalize each cell to target sum (e.g., 1e4 = CP10k)
sc.pp.normalize_total(data, target_sum=1e4)   # in-place on adata.X

# log1p transform (natural log)
sc.pp.log1p(data)

In [ ]:
# Convert gene symbols to ensembl IDs
import mygene
import pandas as pd

# Initialize MyGeneInfo
mg = mygene.MyGeneInfo()

# Get your gene symbols from AnnData
gene_symbols = data.var_names.tolist()

# Query MyGene for Ensembl gene IDs
out = mg.querymany(
    gene_symbols,
    scopes="symbol",      # input type
    fields="ensembl.gene",# output field
    species="human"       # or "mouse" if needed
)

# Convert results into DataFrame for easy mapping
df = pd.DataFrame(out)

# Some genes return multiple Ensembl IDs; keep the first one
df["ensembl_id"] = df["ensembl"].apply(
    lambda x: x[0]["gene"] if isinstance(x, list) else (x["gene"] if isinstance(x, dict) else None)
)

# Build mapping dictionary: {symbol -> ensembl_id}
mapping = df.set_index("query")["ensembl_id"].dropna().to_dict()

# Replace var_names
data.var["gene_symbol"] = data.var_names
data.var_names = [mapping.get(g, g) for g in data.var_names]
data.var_names_make_unique()

## Set up the pipeline

In [ ]:
pipeline = CellEmbeddingPipeline(pretrain_prefix=PRETRAIN_VERSION, # Specify the pretrain checkpoint to load
                                 pretrain_directory='../ckpt')
pipeline.model

## Extract embeddings

In [ ]:
embedding = pipeline.predict(data, # An AnnData object
                device=DEVICE) # Specify a gpu or cpu for model inference

data.obsm['emb'] = embedding.cpu().numpy()
emb = data.obsm['emb']

## Save embeddings

In [ ]:
# Save updated AnnData

output_path = './embeddings/BMMC_with_embeddings.h5ad' 

data.write_h5ad(output_path)

print(f"Embeddings saved to {output_path} in obsm['emb']")

## UMAP Visualisation of Embeddings

If loading embeddings directly, run the import block at top of notebook first.

In [ ]:
# Set a seed for reproducibility

import os
import torch
import random

def fix_seed(seed):
    # Fix for Python hash seed (to ensure reproducibility in Python operations)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix for random seed (used by random module)
    random.seed(seed)
    
    # Fix for numpy random operations
    np.random.seed(seed)
    
    # Fix for PyTorch random seed (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional configuration for CUBLAS for reproducibility
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # For scanpy or any random process in other libraries
    # We can use a random_state argument where applicable, like in scanpy PCA, DEG, etc.

# Set the seed
seed = 42
fix_seed(seed)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Read the AnnData objects
adata = sc.read_h5ad("./embeddings/BMMC_with_zeroshot_embeddings.h5ad")

In [ ]:
# UMAP
sc.pp.neighbors(data, use_rep='emb', random_state=seed)
sc.tl.umap(data, random_state=seed)
plt.rcParams['figure.figsize'] = (6, 6)
sc.pl.umap(data, color='cell_type', palette='Paired', title='CellPLM', show=False)
ax = plt.gca()
handles, labels = ax.get_legend_handles_labels()
plt.legend(
    handles, labels,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    ncol=1,  # <-- this forces single-column legend
    fontsize='small',
    frameon=False
)
file_path = './figures/umap_zero_shot_clustering_BMMC.svg'
plt.savefig(file_path, dpi=500, bbox_inches='tight')
plt.show()

## Clustering

In [ ]:
import numpy as np
import pandas as pd
from sklearn import metrics
import scanpy as sc
from sklearn.decomposition import PCA


def clustering(adata, n_clusters=7, key='emb', method='leiden', start=0.1, end=3.0, increment=0.01):
    """\
    Spatial clustering based the learned representation.

    Returns
    -------
    None.

    """
    
    pca = PCA(n_components=20, random_state=seed) 
    embedding = pca.fit_transform(adata.obsm[key].copy())
    adata.obsm['emb_pca'] = embedding
    
    if method == 'leiden':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.leiden(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['leiden']
    elif method == 'louvain':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.louvain(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['louvain'] 
       
    

def extract_top_value(map_matrix, retain_percent = 0.1): 
    '''\
    Filter out cells with low mapping probability

    Parameters
    ----------
    map_matrix : array
        Mapped matrix with m spots and n cells.
    retain_percent : float, optional
        The percentage of cells to retain. The default is 0.1.

    Returns
    -------
    output : array
        Filtered mapped matrix.

    '''

    #retain top 1% values for each spot
    top_k  = retain_percent * map_matrix.shape[1]
    output = map_matrix * (np.argsort(np.argsort(map_matrix)) >= map_matrix.shape[1] - top_k)
    
    return output 
    
def search_res(adata, n_clusters, method='leiden', use_rep='emb', start=0.1, end=3.0, increment=0.01):
    '''\
    Searching corresponding resolution according to given cluster number
    
    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Targetting number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.    
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float 
        The end value for searching.
    increment : float
        The step size to increase.
        
    Returns
    -------
    res : float
        Resolution.
        
    '''
    print('Searching resolution...')
    label = 0
    sc.pp.neighbors(adata, n_neighbors=50, use_rep=use_rep, random_state=seed)
    for res in sorted(list(np.arange(start, end, increment)), reverse=True):
        if method == 'leiden':
           sc.tl.leiden(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['leiden']).leiden.unique())
           print('resolution={}, cluster number={}'.format(res, count_unique))
        elif method == 'louvain':
           sc.tl.louvain(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['louvain']).louvain.unique()) 
           print('resolution={}, cluster number={}'.format(res, count_unique))
        if count_unique == n_clusters:
            label = 1
            break

    assert label==1, "Resolution is not found. Please try bigger range or smaller step!." 
       
    return res    


In [ ]:
# Run Leiden clustering 

n_clusters = 45
tool='leiden'

clustering(adata, n_clusters, key='emb', method=tool, start=0.1, end=3, increment=0.01)

In [ ]:
labels = adata.obs['leiden'].astype(int)

## ARI, NMI & Silhouette Scores

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

ari_score = adjusted_rand_score(adata.obs['leiden'].to_numpy(), adata.obs['cell_type'].to_numpy())
nmi_score = normalized_mutual_info_score(adata.obs['leiden'].to_numpy(), adata.obs['cell_type'].to_numpy())
sil_score = silhouette_score(adata.obsm['emb'], adata.obs['leiden'].astype(int))

print(f"'ari': {ari_score}, 'nmi': {nmi_score}, 'sil': {sil_score}")

## Save results to npz file

In [ ]:
Model_name='cellplm'
step='zero_shot'
dataset='BMMC'

In [ ]:
import numpy as np

ARI, NMI, SIL = float(ari_score), float(nmi_score), float(sil_score)

np.savez_compressed(
    f"./benchmarking_results/{Model_name}_{step}_clusters_{dataset}.npz",
    labels=adata.obs['cell_type'],      
    embeddings=emb,
    ARI=ARI, NMI=NMI, SIL=SIL,
)